# 03 - Train a DQN Model Online

This notebook shows the online training workflow in Mouse Core. Instead of loading a fixed dataset, it keeps live environments in the loop:

1. Build train and held-out eval `GroupEnv`s.
2. For each of `NUM_CYCLES` cycles: collect `ROLLOUT_STEPS` env steps, run `TRAIN_STEPS` optimizer updates, then `EVAL_STEPS` eval tasks.
3. Append transitions into in-memory `Datastore` replay streams.
4. Sample those streams with `DataLoader`.
5. Train with `DqnObjective`.

The important Mouse Core idea is that online and offline training use the same row format, model interface, datastores, dataloader, and objective. Only the source of rows changes.

Rollouts use **train** environments. A separate **eval** `GroupEnv` (offset seeds) is scored greedily each cycle so reported task scores are not polluted by exploration noise.


In [ ]:
from collections import deque

import torch
import numpy as np

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)

from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead
from mouse_core.objectives import DqnObjective


MODEL_ID = "mouse-example-model-online"       # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
MAX_EPISODE_STEPS = 30                        # environment-specific episode limit
EPISODES_PER_TASK = 20                        # environment-specific task length
NUM_ENVS = 30                                 # number of environment streams in the GroupEnv

SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step

NUM_CYCLES = 100                              # outer rollout/train/eval cycles
ROLLOUT_STEPS = 500                           # env steps per cycle (passed to run_rollout)
TRAIN_STEPS = 200                             # optimizer updates per cycle (passed to run_train)
EVAL_STEPS = 1                                # tasks per eval env per cycle (passed to run_eval)

EVAL_NUM_ENVS = 4
EVAL_SEED_OFFSET = 1_000_000                  # held-out env seed stream (far from train)

LEARNING_STARTS = 15_000                      # replay rows collected before the first optimizer update
EXPLORATION_ENDS = 1_500_000                  # env-step horizon for epsilon decay

# Rollout rows are appended to datastores and become replay samples after loader.refresh().


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Environment

Online training uses the same `EnvConfig` and `make_group_env` pattern as data collection. Build the **train** `GroupEnv` for rollouts and a separate **eval** `GroupEnv` (seed stream offset by `EVAL_SEED_OFFSET`) for greedy scoring so reported task scores are not polluted by exploration noise.

The group environment steps all configured instances together and returns one output dictionary per instance. Keep row fields consistent with the model and objective: this notebook consumes `action`, `observation`, `reward`, and `done`; the DQN objective also uses `done` to decide whether bootstrapping should continue across a boundary.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # forwarded to the environment at task reset
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i,
            "slippery_success_rate": 1.0,  # environment-specific option
            "permute_obs": True,      # environment-specific option
            "permute_actions": True,  # environment-specific option
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)


In [ ]:
eval_configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"eval_frozenlake_{i}",
        reset_seed=i + EVAL_SEED_OFFSET,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i + EVAL_SEED_OFFSET,
            "slippery_success_rate": 1.0,
            "permute_obs": True,
            "permute_actions": True,
        },
    )
    for i in range(EVAL_NUM_ENVS)
]

eval_env = make_group_env(eval_configs)


## Build The Model

This is the same model assembly pattern used by offline training: `NumericEmbedder` for row fields, `Qwen3Backbone` for sequence processing, and `DiscreteActionValueHead` for action values.

Because the online policy calls the model during rollout, the same model object is used in two modes: `eval()` for action selection and `train()` for optimizer updates.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device=device, dtype=preferred_dtype(device))
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)


## Replay Buffer And Policy Contexts

Each environment stream writes to one `Datastore`; together they act as the replay buffer. Each environment also keeps a bounded `contexts` deque containing the recent rows used for action selection.

The replay buffer is for training batches. The context deque is for policy inference during rollout. Keeping them separate makes it clear which history is sampled for learning and which history conditions the next action.


In [ ]:
stores = [Datastore(name=name) for name in env.names]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in env.names]


## Evaluation Phase

`run_eval` rolls out the current policy greedily on `eval_env`. It does not append to train replay. Row **contexts persist** across calls so evaluation continues mid-task; the **KV cache does not** and is rebuilt from those contexts at the start of each call. Each outer cycle passes `num_steps=EVAL_STEPS` (tasks per eval env).


In [ ]:
eval_contexts = [[] for _ in eval_env.names]

def run_eval(
    *,
    model,
    env,
    contexts: list,
    num_steps: int,
    max_cache: int = 512,
    start_cache=None,
    temperature: float = 0.0,
):
    """Greedy task-budget eval on a GroupEnv (does not append to train replay).

    ``num_steps`` is the number of tasks to complete per env stream.
    ``contexts`` persist across calls (mutated in place); the KV cache does not
    and is rebuilt from ``contexts`` at the start of each call.
    """
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache, start_cache)
    n = len(env.names)
    if len(contexts) != n:
        raise ValueError(f"contexts has {len(contexts)} streams but env has {n}")
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    eval_task_scores = [[] for _ in range(n)]
    inputs = None
    outputs = None
    env.metrics.clear()
    num_actions = env.action_space.spaces[0].n

    def _act_from_contexts(*, rebuild: bool) -> list[dict]:
        nonlocal kv_cache, cached_starts, cached_ends
        ends = np.array([len(c) for c in contexts], dtype=np.int64)
        need_rebuild = rebuild or cache_needs_rebuild(
            has_cache=kv_cache is not None,
            cached_starts=cached_starts,
            cached_ends=cached_ends,
            ends=ends,
            context_start=context_start,
            max_cache=max_cache,
            batch_complete=True,
        )
        with torch.no_grad():
            if need_rebuild:
                starts = rebuild_starts(
                    ends=ends,
                    context_start=context_start,
                    start_cache=start_cache,
                    max_cache=max_cache,
                )
                batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                predictions, _, kv_cache = model(batch, use_cache=True)
                cached_starts = starts
                cached_ends = ends.copy()
            else:
                batch = [
                    contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)
                ]
                predictions, _, kv_cache = model(batch, cache=kv_cache, use_cache=True)
                cached_ends = ends.copy()
            actions = model.get_action(
                predictions, temperature=temperature, num_actions=num_actions
            )
        random_inputs = env.sample_random_input()
        return [
            {"action": action} if contexts[i] else random_inputs[i]
            for i, action in enumerate(actions.cpu().numpy())
        ]

    while min(len(scores) for scores in eval_task_scores) < num_steps:
        if outputs is None:
            # Start of this call: rebuild KV from persisted contexts (or random if empty).
            if any(contexts):
                inputs = _act_from_contexts(rebuild=True)
            else:
                inputs = env.sample_random_input()
        else:
            task_ended = False
            for i, (inp, out) in enumerate(zip(inputs, outputs)):
                row = {**inp, **out}
                row.pop("info", None)
                contexts[i].append(row)
                if int(row["done"]) >= 3:
                    contexts[i] = []
                    context_start[i] = 0
                    task_ended = True
            inputs = _act_from_contexts(rebuild=task_ended)
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            if int(out["done"]) >= 3:
                eval_task_scores[i].append(float(env.metrics.task_cum_rewards[i][-1]))

    scores_per_env = [scores[:num_steps] for scores in eval_task_scores]
    flat_scores = [s for scores in scores_per_env for s in scores]
    mean_task_score = (
        sum(flat_scores) / len(flat_scores) if flat_scores else float("nan")
    )
    return {
        "mean_task_score": mean_task_score,
        "task_scores": flat_scores,
        "scores_per_env": scores_per_env,
        "n_tasks": len(flat_scores),
    }


## Rollout Phase

A rollout collects `num_steps` lockstep env steps (callers pass `ROLLOUT_STEPS`) from the live environments. The loop follows the same shape each step:

1. Decide which environments should act greedily and which should explore randomly.
2. For greedy environments, call `model(..., use_cache=True)` on pending context rows.
3. Convert predictions to actions with `model.get_action(...)`.
4. Step the `GroupEnv` with one input dictionary per environment.
5. Append each merged row to its datastore, context deque, and pending decode chunk.

`use_cache=True` returns a cache object that can be passed back into the next model call. This avoids recomputing the whole context on every action selection. The cache is local to a rollout; the next rollout rebuilds from the bounded context deques.

Row **contexts persist** across calls; the **KV cache does not** and is rebuilt from those contexts at the start of each call.


In [ ]:
def epsilon_for_env_step(*, env_step: int) -> float:
    """Return the epsilon-greedy exploration rate for a global env step."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [ ]:
def run_rollout(*, model: Model, env, stores: list[Datastore], contexts: list[deque], env_steps: int, num_steps: int) -> int:
    """Collect ``num_steps`` lockstep env steps and append rows to replay datastores.

    ``contexts`` persist across calls; the KV cache does not and is rebuilt
    from ``contexts`` at the start of each call.
    """
    env.metrics.clear()
    model.eval()
    kv_cache = None
    pending_rows = [list(c) for c in contexts]
    for step in range(num_steps):
        epsilon = epsilon_for_env_step(env_step=env_steps)
        greedy = [bool(c) and torch.rand(1).item() >= epsilon for c in contexts]
        if any(greedy):
            with torch.no_grad():
                predictions, _, kv_cache = model(pending_rows, cache=kv_cache, use_cache=True)
            actions = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
            pending_rows = [[] for _ in contexts]
        random_inputs = env.sample_random_input()
        inputs = [{'action': actions[i].cpu().numpy()} if greedy[i] else random_inputs[i] for i in range(len(contexts))]
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            row = {**inputs[i], **out}
            row.pop('info', None)
            stores[i].append(row)
            contexts[i].append(row)
            pending_rows[i].append(row)
        env_steps += len(outputs)
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return env_steps


## Training Phase

After a rollout, training uses the same API as offline training. `loader.refresh()` tells the dataloader to rescan the growing datastores and clear prefetched batches, then `run_train` runs `num_steps=TRAIN_STEPS` DQN updates.

`sequence_length` is a max window length — short stores are fine. `weight_mode="per_step"` samples in proportion to the amount of data in each store.


In [ ]:
loader = DataLoader(stores=stores, sequence_length=SEQUENCE_LENGTH, batch_size=BATCH_SIZE, weight_mode='per_step', num_workers=0)
objective = DqnObjective(
    gamma_step=1.0,                 # discount for ordinary next-step bootstrapping
    gamma_episode_terminal=1.0,     # discount when an episode ends normally
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # discount when a task ends normally
    gamma_task_truncated=0.0,
    tau=0.0005,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

def run_train(
    *,
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: DqnObjective,
    loader: DataLoader,
    num_steps: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``num_steps`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)

    assert loss is not None
    return loss, metrics


## Run Online Training

The main loop runs `NUM_CYCLES` times. Each cycle calls `run_rollout(num_steps=ROLLOUT_STEPS)`, then `run_train(num_steps=TRAIN_STEPS)` once replay has at least `LEARNING_STARTS` rows, then `run_eval(num_steps=EVAL_STEPS)`.


In [ ]:
env_steps = 0
grad_steps = 0
for cycle in range(NUM_CYCLES):
    env_steps = run_rollout(model=model, env=env, stores=stores, contexts=contexts, env_steps=env_steps, num_steps=ROLLOUT_STEPS)
    print(f"cycle={cycle} rollout  env_step={env_steps}")
    if env_steps >= LEARNING_STARTS:
        loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
        grad_steps += TRAIN_STEPS
        print(f"cycle={cycle} train    loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}  grad_step={grad_steps}")
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    stats = run_eval(num_steps=EVAL_STEPS, max_cache=SEQUENCE_LENGTH, model=model, env=eval_env, contexts=eval_contexts)
    print(f"cycle={cycle} eval     mean_task_score={stats['mean_task_score']:.2f}/{EPISODES_PER_TASK}  n_tasks={stats['n_tasks']}")
loader.close()
env.close()
eval_env.close()
print(f'Online training finished ({grad_steps} optimizer steps, {env_steps} env steps).')


## Push To The Hub

Run this cell to save the online-trained checkpoint. The inference notebook can load it later with `load_model`.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")